In [1]:
# Dataverse Connection Configuration
dataverse_url = "https://rt-citizen-prime-prd-au.crm6.dynamics.com"
client_id = "c249f153-f8f1-44c5-8c3b-5e7368635f8d"

# TODO: Provide these values
tenant_id = "4341df80-fbe6-41bf-89b0-e6e2379c9c23"  # Azure AD Tenant ID
client_secret = os.environ["DATAVERSE_CLIENT_SECRET"] #dbutils.secrets.get(scope="<SECRET_SCOPE>", key="<SECRET_KEY>")  # Or paste directly (not recommended)

# Acquire OAuth2 token
import requests

token_url = f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token"
token_payload = {
    "grant_type": "client_credentials",
    "client_id": client_id,
    "client_secret": client_secret,
    "scope": f"{dataverse_url}/.default"
}

token_response = requests.post(token_url, data=token_payload)
token_response.raise_for_status()
access_token = token_response.json()["access_token"]
print("Successfully authenticated to Dataverse.")

Successfully authenticated to Dataverse.


In [ ]:
import os
import socket
import ssl
import requests
import pandas as pd
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Dataverse metadata endpoint
dataverse_url = "https://rt-citizen-prime-prd-au.crm6.dynamics.com"
entities_url = f"{dataverse_url}/api/data/v9.2/EntityDefinitions"

headers = {
    "Authorization": f"Bearer {access_token}",
    "Accept": "application/json",
    "OData-Version": "4.0",
    "OData-MaxVersion": "4.0"
}

# Optional proxy support (set these in cluster environment/init script if required)
# os.environ["HTTPS_PROXY"] = "http://proxy.company:8080"
# os.environ["HTTP_PROXY"] = "http://proxy.company:8080"
proxies = None
if os.getenv("HTTPS_PROXY") or os.getenv("HTTP_PROXY"):
    proxies = {
        "http": os.getenv("HTTP_PROXY"),
        "https": os.getenv("HTTPS_PROXY")
    }


def tls_preflight(host: str, port: int = 443, timeout: int = 8):
    """Check DNS, TCP reachability, and TLS handshake for a host."""
    result = {"host": host, "dns": False, "tcp": False, "tls": False, "details": ""}
    try:
        ip = socket.gethostbyname(host)
        result["dns"] = True
        with socket.create_connection((host, port), timeout=timeout) as sock:
            result["tcp"] = True
            ctx = ssl.create_default_context()
            with ctx.wrap_socket(sock, server_hostname=host) as ssock:
                result["tls"] = True
                result["details"] = f"ip={ip}, tls={ssock.version()}"
    except Exception as ex:
        result["details"] = str(ex)
    return result


# Preflight both auth and data endpoints to prove where path breaks on Shared Compute
preflight_hosts = ["login.microsoftonline.com", "rt-citizen-prime-prd-au.crm6.dynamics.com"]
preflight_results = [tls_preflight(h) for h in preflight_hosts]
print("Network preflight:")
for r in preflight_results:
    print(f"  {r['host']}: dns={r['dns']} tcp={r['tcp']} tls={r['tls']} ({r['details']})")

# Retry for transient network errors and throttling
session = requests.Session()
retry = Retry(
    total=5,
    connect=5,
    read=5,
    backoff_factor=1.0,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"]
)
adapter = HTTPAdapter(max_retries=retry)
session.mount("https://", adapter)
session.mount("http://", adapter)

try:
    response = session.get(
        entities_url,
        headers=headers,
        timeout=(10, 90),
        proxies=proxies
    )

    if response.status_code >= 400:
        print(f"HTTP {response.status_code}: {response.reason}")
        try:
            print("Error body:", response.json())
        except ValueError:
            print("Error body (text):", response.text)

    response.raise_for_status()
    entities = response.json().get("value", [])

except requests.exceptions.SSLError as e:
    print(f"SSL handshake error: {e}")
    print("Likely cause on Shared Compute: outbound TLS path is blocked/intercepted.")
    print("Proxy env detected:", bool(os.getenv("HTTPS_PROXY") or os.getenv("HTTP_PROXY")))
    print("Next steps:")
    print("  1. Ask platform/network team to allow outbound 443 to rt-citizen-prime-prd-au.crm6.dynamics.com")
    print("  2. Confirm TLS interception/proxy trust is configured on this compute")
    print("  3. If Shared Compute cannot use required egress/proxy, run on single-user compute")
    raise

except requests.exceptions.RequestException as e:
    print(f"Request error: {e}")
    raise

finally:
    session.close()

print(f"Found {len(entities)} entities in Dataverse\\n")

entity_list = []
for e in entities:
    display_name = ""
    if e.get("DisplayName") and e["DisplayName"].get("UserLocalizedLabel"):
        display_name = e["DisplayName"]["UserLocalizedLabel"].get("Label", "")

    entity_set_name = e.get("EntitySetName")
    if not entity_set_name:
        continue

    entity_list.append({
        "LogicalName": e.get("LogicalName"),
        "DisplayName": display_name,
        "EntitySetName": entity_set_name,
        "IsCustomEntity": e.get("IsCustomEntity")
    })

# Client-side sorting to replace rejected $orderby
entity_list = sorted(entity_list, key=lambda x: (x["LogicalName"] or ""))

df_entities = pd.DataFrame(entity_list)
print(df_entities.head(10))

Found 937 entities in Dataverse\n
              LogicalName                DisplayName  \
0                 aaduser         Microsoft Entra ID   
1                 account                    Account   
2           aciviewmapper              ACIViewMapper   
3              actioncard                Action Card   
4  actioncardusersettings  Action Card User Settings   
5     actioncarduserstate        ActionCardUserState   
6  activityfileattachment   Activity File Attachment   
7  activitymimeattachment                 Attachment   
8           activityparty             Activity Party   
9         activitypointer                   Activity   

               EntitySetName  IsCustomEntity  
0                   aadusers            True  
1                   accounts           False  
2             aciviewmappers           False  
3                actioncards           False  
4  actioncardusersettingsset           False  
5       actioncarduserstates           False  
6    activityfileatt

In [8]:
df_entities

,LogicalName,DisplayName,EntitySetName,IsCustomEntity
0,aaduser,Microsoft Entra ID,aadusers,True
1,account,Account,accounts,False
2,aciviewmapper,ACIViewMapper,aciviewmappers,False
3,actioncard,Action Card,actioncards,False
4,actioncardusersettings,Action Card User Settings,actioncardusersettingsset,False
...,...,...,...,...
932,workflowlog,Process Log,workflowlogs,False
933,workflowmetadata,Workflow Metadata,workflowmetadata,True
934,workflowwaitsubscription,Workflow Wait Subscription,workflowwaitsubscriptions,False
935,workqueue,Work Queue,workqueues,True
